# Feature Engineering — Dữ liệu Ngoài (External Features)

| Nhóm | Feature | Mô tả |
|---|---|---|
| Oil | `oil_price` | Giá dầu WTI đã ffill |
| Oil | `oil_price_lag_7/14` | Lag 7/14 ngày |
| Oil | `oil_price_rolling_mean_28` | Rolling mean 28 ngày (shift-1 safe) |
| Oil | `oil_price_change_pct` | % thay đổi vs tuần trước |
| Store | `cluster` | Cluster số (1–17, dùng trực tiếp) |
| Store | `store_type_encoded` | A→0, B→1, C→2, D→3, E→4 |
| Store | `city_freq`, `state_freq` | Frequency encoding vị trí |

In [1]:
import pandas as pd
import numpy as np

In [ ]:
import sys
from pathlib import Path

def _find_root():
    """Tìm project root chứa config.py."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'config.py').exists():
            return p
    raise RuntimeError("Không tìm thấy project root. Mở Jupyter từ thư mục gốc của project.")

_root = _find_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import PROCESSED_DIR

df_train  = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')
df_oil    = pd.read_csv(PROCESSED_DIR / 'cleaned_oil.csv')
df_stores = pd.read_csv(PROCESSED_DIR / 'stores_cleaned.csv')

print(f'Train shape  : {df_train.shape}')
print(f'Oil shape    : {df_oil.shape}')
print(f'Stores shape : {df_stores.shape}')

---
## 1. Oil Price Features

In [ ]:
def add_oil_features(
    df: pd.DataFrame,
    oil_df: pd.DataFrame,
    date_col: str = "date",
) -> pd.DataFrame:

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    oil = oil_df.copy()
    oil = oil.rename(columns={"dcoilwtico": "oil_price"})
    oil["date"] = pd.to_datetime(oil["date"])
    oil = oil[["date", "oil_price"]].sort_values("date").reset_index(drop=True)

    # Tạo dải ngày liên tục để không bỏ sót cuối tuần/ngày lễ
    full_date_range = pd.DataFrame({
        "date": pd.date_range(start=oil["date"].min(), end=oil["date"].max(), freq="D")
    })
    oil = full_date_range.merge(oil, on="date", how="left")

    oil["oil_price"] = oil["oil_price"].ffill()
    oil["oil_price"] = oil["oil_price"].bfill()

    oil["oil_price_lag_7"]  = oil["oil_price"].shift(7)
    oil["oil_price_lag_14"] = oil["oil_price"].shift(14)

    # shift(1) — rolling window không bao gồm ngày hiện tại (consistent với build_train_final.py)
    oil["oil_price_rolling_mean_28"] = (
        oil["oil_price"].shift(1).rolling(window=28, min_periods=7).mean()
    )

    oil["oil_price_change_pct"] = oil["oil_price"].pct_change(periods=7)

    oil_cols = [
        "date", "oil_price", "oil_price_lag_7", "oil_price_lag_14",
        "oil_price_rolling_mean_28", "oil_price_change_pct",
    ]
    df = df.merge(oil[oil_cols], on="date", how="left")
    return df

---
## 2. Store & Product Encoding

In [4]:
def add_store_encoding(
    df: pd.DataFrame,
    stores_df: pd.DataFrame,
) -> pd.DataFrame:

    df = df.copy()
    stores = stores_df.copy()

    # Bước 1: Label Encoding cho store_type (A, B, C, D, E → 0, 1, 2, 3, 4)
    # store_type chỉ có 5 loại duy nhất → Label Encoding đơn giản là đủ.
    # Chuyển A→0, B→1, ... theo thứ tự alphabet (nhất quán, không ngẫu nhiên).
    type_mapping = {t: i for i, t in enumerate(sorted(stores["type"].unique()))}
    stores["store_type_encoded"] = stores["type"].map(type_mapping)

    # Bước 2: Frequency Encoding cho city và state
    # Frequency encoding: thay tên thành phố/tỉnh bằng xác suất xuất hiện
    # Các thành phố lớn có nhiều cửa hàng → giá trị freq cao hơn
    # → model hiểu được quy mô địa lý một cách ngầm định

    city_freq  = stores["city"].value_counts(normalize=True).to_dict()
    state_freq = stores["state"].value_counts(normalize=True).to_dict()

    stores["city_freq"]  = stores["city"].map(city_freq)
    stores["state_freq"] = stores["state"].map(state_freq)

    # Bước 3: Giữ cluster nguyên (đã là số)
    # cluster là số từ 1-17, LightGBM xử lý trực tiếp được.
    # Không cần encode thêm.

    # Bước 4: Merge vào DataFrame chính theo store_nbr
    store_cols = [
        "store_nbr",
        "cluster",            # giữ nguyên
        "store_type_encoded", # A-E → 0-4
        "city_freq",          # frequency của thành phố
        "state_freq",         # frequency của tỉnh/bang
    ]
    df = df.merge(stores[store_cols], on="store_nbr", how="left")

    return df

---
## 3. Pipeline Tổng hợp

In [5]:
def build_external_features(
    df: pd.DataFrame,
    oil_df: pd.DataFrame,
    stores_df: pd.DataFrame,
    date_col: str = "date",
) -> pd.DataFrame:

    df = add_oil_features(df, oil_df, date_col=date_col)
    df = add_store_encoding(df, stores_df)
    return df

---
## 4. Feature Name Registry

In [6]:
OIL_FEATURE_NAMES = [
    "oil_price",
    "oil_price_lag_7",
    "oil_price_lag_14",
    "oil_price_rolling_mean_28",
    "oil_price_change_pct",
]

STORE_FEATURE_NAMES = [
    "cluster",
    "store_type_encoded",
    "city_freq",
    "state_freq",
]

ALL_EXTERNAL_FEATURE_NAMES = OIL_FEATURE_NAMES + STORE_FEATURE_NAMES

print(f'Oil features   : {OIL_FEATURE_NAMES}')
print(f'Store features : {STORE_FEATURE_NAMES}')
print(f'Total          : {len(ALL_EXTERNAL_FEATURE_NAMES)} features')

Oil features   : ['oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct']
Store features : ['cluster', 'store_type_encoded', 'city_freq', 'state_freq']
Total          : 9 features


---
## 5. Chạy Pipeline trên Dữ liệu Thực

In [7]:
df_featured = build_external_features(df_train, df_oil, df_stores)

print(f'Input shape  : {df_train.shape}')
print(f'Output shape : {df_featured.shape}')
print(f'New columns  : {df_featured.shape[1] - df_train.shape[1]}')
print()
print('=== Feature Summary ===')
df_featured[ALL_EXTERNAL_FEATURE_NAMES].describe().T[['mean', 'min', 'max']]

Input shape  : (3000888, 6)
Output shape : (3000888, 15)
New columns  : 9

=== Feature Summary ===


,mean,min,max
oil_price,67.924899,26.190000,110.620000
oil_price_lag_7,68.009296,26.190000,110.620000
oil_price_lag_14,68.083719,26.190000,110.620000
oil_price_rolling_mean_28,68.279963,30.210000,108.076429
oil_price_change_pct,-0.001698,-0.171989,0.287284
cluster,8.481481,1.000000,17.000000
store_type_encoded,2.000000,0.000000,4.000000
city_freq,0.149520,0.018519,0.333333
state_freq,0.182442,0.018519,0.351852


---
## 6. Smoke Test

In [8]:
print("=" * 60)
print("SMOKE TEST: feature_external")
print("=" * 60)

# --- Tạo data giả để test ---
np.random.seed(42)
dates = pd.date_range("2016-01-01", periods=90, freq="D")

# DataFrame chính (train giả)
mock_train = pd.DataFrame({
    "date":      list(dates) * 3,
    "store_nbr": [1] * 90 + [2] * 90 + [3] * 90,
    "family":    ["GROCERY I"] * 270,
    "sales":     np.random.rand(270) * 1000,
})

# Oil DataFrame giả (thêm vài NaN để test ffill)
mock_oil = pd.DataFrame({
    "date":        pd.date_range("2015-12-01", periods=120, freq="D"),
    "dcoilwtico":  np.where(
        np.random.rand(120) < 0.15,  # 15% ngày thiếu giá (cuối tuần/lễ)
        np.nan,
        50 + np.random.randn(120) * 5  # giá dầu ~50 USD
    ),
})

# Stores DataFrame giả
mock_stores = pd.DataFrame({
    "store_nbr": [1, 2, 3],
    "type":      ["A", "B", "A"],
    "cluster":   [1, 3, 1],
    "city":      ["Quito", "Guayaquil", "Quito"],
    "state":     ["Pichincha", "Guayas", "Pichincha"],
})

# --- Chạy pipeline ---
result = build_external_features(mock_train, mock_oil, mock_stores)

print(f"\nShape đầu vào:  {mock_train.shape}")
print(f"Shape đầu ra:   {result.shape}")

print("\n--- Kiểm tra từng feature ---")
all_ok = True
for col in ALL_EXTERNAL_FEATURE_NAMES:
    if col in result.columns:
        null_pct = result[col].isna().mean() * 100
        sample_val = result[col].dropna().iloc[0] if not result[col].dropna().empty else "N/A"
        print(f"  ✓  {col:<35}  NaN={null_pct:5.1f}%  sample={sample_val:.4f}" if isinstance(sample_val, float) else f"  ✓  {col:<35}  NaN={null_pct:5.1f}%  sample={sample_val}")
    else:
        print(f"  ✗  {col:<35}  MISSING!")
        all_ok = False

# --- Kiểm tra oil_price không còn NaN sau ffill ---
oil_nan = result["oil_price"].isna().sum()
print(f"\nSau ffill, oil_price NaN còn lại: {oil_nan} (phải = 0 hoặc rất nhỏ)")

# --- Kiểm tra store_type_encoded ---
unique_types = result["store_type_encoded"].unique()
print(f"store_type_encoded unique values: {sorted(unique_types)}")

# --- Kiểm tra city_freq trong khoảng [0, 1] ---
assert result["city_freq"].dropna().between(0, 1).all(), "city_freq ngoài khoảng [0,1]!"
print("city_freq trong khoảng [0, 1]: ✓")

# --- Kiểm tra lag_14 có giá trị sau 14 ngày đầu ---
lag14_after_warmup = result[result["date"] >= dates[14]]["oil_price_lag_14"]
nan_after = lag14_after_warmup.isna().sum()
print(f"oil_price_lag_14 NaN sau warmup 14 ngày: {nan_after} (nên thấp)")

print("\n" + ("✅ TẤT CẢ TESTS PASSED!" if all_ok else "❌ CÓ LỖI, kiểm tra lại!"))
print("\n--- Xem trước 3 dòng đầu ---")
print(result[["date", "store_nbr"] + ALL_EXTERNAL_FEATURE_NAMES].head(3).to_string())

SMOKE TEST: feature_external

Shape đầu vào:  (270, 4)
Shape đầu ra:   (270, 13)

--- Kiểm tra từng feature ---
  ✓  oil_price                            NaN=  1.1%  sample=49.2647
  ✓  oil_price_lag_7                      NaN=  1.1%  sample=53.9583
  ✓  oil_price_lag_14                     NaN=  1.1%  sample=52.9758
  ✓  oil_price_rolling_mean_28            NaN=  1.1%  sample=52.6599
  ✓  oil_price_change_pct                 NaN=  1.1%  sample=-0.0870
  ✓  cluster                              NaN=  0.0%  sample=1
  ✓  store_type_encoded                   NaN=  0.0%  sample=0
  ✓  city_freq                            NaN=  0.0%  sample=0.6667
  ✓  state_freq                           NaN=  0.0%  sample=0.6667

Sau ffill, oil_price NaN còn lại: 3 (phải = 0 hoặc rất nhỏ)
store_type_encoded unique values: [np.int64(0), np.int64(1)]
city_freq trong khoảng [0, 1]: ✓
oil_price_lag_14 NaN sau warmup 14 ngày: 3 (nên thấp)

✅ TẤT CẢ TESTS PASSED!

--- Xem trước 3 dòng đầu ---
        date  stor

In [ ]:
# CSV write disabled — tất cả features được merge trong build_train_final.py.
# Để tạo CSV cho model training:
#   python scripts/build_train_final.py   →  data/processed/train_final.csv
#   python scripts/create_splits.py       →  data/processed/splits/
print('[SKIPPED] CSV write disabled')
print(f'Feature dataframe shape : {df_featured.shape}')
print(f'Columns: {df_featured.columns.tolist()}')